In [14]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
import itertools
import matplotlib.pyplot as plt

In [15]:
prices = pd.read_csv(
    "./dataset/cleaned_data.csv", parse_dates=["Date"], index_col="Date"
)

print("Price matrix:", prices.shape)
display(prices.head())

Price matrix: (3905, 432)


,GNRC,CHTR,MTCH,NDAQ,WFC,WM,WELL,WMB,MU,NDSN,...,EXPE,EXPD,EXC,EW,EVRG,ETR,ETN,ESS,ES,IT
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,12.84,35.0,5.856653,6.746667,27.320000,34.160000,44.040001,17.616472,10.85,31.530001,...,51.639999,35.080002,34.864479,7.289167,21.840000,41.075001,32.160000,82.620003,25.770000,18.700001
2010-01-05,12.84,35.0,5.985151,6.766667,28.070000,34.009998,44.660000,17.836576,11.17,31.490000,...,51.860001,35.320000,34.293865,7.341667,21.549999,40.419998,31.969999,83.120003,25.680000,18.730000
2010-01-06,12.84,35.0,5.936608,6.763333,28.110001,34.000000,44.439999,18.415367,11.22,31.139999,...,49.200001,34.500000,34.500713,7.422500,21.670000,40.625000,31.830000,83.699997,26.010000,18.860001
2010-01-07,12.84,35.0,5.965163,6.673333,29.129999,34.080002,44.529999,18.284937,10.84,31.514999,...,48.939999,34.290001,34.614838,7.490000,21.530001,40.139999,32.299999,84.570000,25.969999,20.440001
2010-01-08,12.84,33.5,6.002284,6.743333,28.860001,34.240002,44.070000,18.431671,11.10,31.875000,...,48.480000,34.650002,34.450787,7.456667,21.719999,39.755001,33.025002,83.699997,26.040001,20.660000


In [16]:
# ============================================================================
# CREATE COMPREHENSIVE SECTOR MAPPING FOR ALL STOCKS
# Combines S&P 500 GICS sector classification with general knowledge
# ============================================================================

# Get all ticker symbols from the prices dataframe
all_tickers = prices.columns.tolist()

# Comprehensive GICS Sector Mapping (11 GICS Sectors)
# Based on S&P 500 standard classifications and general knowledge
SECTOR_MAPPING = {
    # INFORMATION TECHNOLOGY
    'AAPL': 'Information Technology', 'MSFT': 'Information Technology', 'NVDA': 'Information Technology',
    'ORCL': 'Information Technology', 'CSCO': 'Information Technology', 'CRM': 'Information Technology',
    'ADBE': 'Information Technology', 'AVGO': 'Information Technology', 'INTC': 'Information Technology',
    'AMD': 'Information Technology', 'AMAT': 'Information Technology', 'KLAC': 'Information Technology',
    'LRCX': 'Information Technology', 'ANSS': 'Information Technology', 'CDNS': 'Information Technology',
    'FTNT': 'Information Technology', 'ITW': 'Information Technology', 'ADSK': 'Information Technology',
    'ADI': 'Information Technology', 'AKAM': 'Information Technology', 'APH': 'Information Technology',
    'HPQ': 'Information Technology', 'IBM': 'Information Technology', 'INTU': 'Information Technology',
    'JKHY': 'Information Technology', 'MSI': 'Information Technology', 'MU': 'Information Technology',
    'MCHP': 'Information Technology', 'MPWR': 'Information Technology', 'NTAP': 'Information Technology',
    'ON': 'Information Technology', 'PAYX': 'Information Technology', 'QCOM': 'Information Technology',
    'SNPS': 'Information Technology', 'SMCI': 'Information Technology', 'SWKS': 'Information Technology',
    'TDY': 'Information Technology', 'TEL': 'Information Technology', 'TER': 'Information Technology',
    'TXN': 'Information Technology', 'TYL': 'Information Technology', 'VRSN': 'Information Technology',
    'ZBRA': 'Information Technology', 'PTC': 'Information Technology', 'STE': 'Information Technology',
    'STX': 'Information Technology', 'TRMB': 'Information Technology', 'NDSN': 'Information Technology',
    
    # COMMUNICATION SERVICES
    'GOOGL': 'Communication Services', 'GOOG': 'Communication Services', 'META': 'Communication Services',
    'NFLX': 'Communication Services', 'T': 'Communication Services', 'VZ': 'Communication Services',
    'TMUS': 'Communication Services', 'CHTR': 'Communication Services', 'CMCSA': 'Communication Services',
    'DIS': 'Communication Services', 'EA': 'Communication Services', 'TTWO': 'Communication Services',
    'LYV': 'Communication Services', 'OMC': 'Communication Services', 'PARA': 'Communication Services',
    'WBD': 'Communication Services', 'MTCH': 'Communication Services',
    
    # CONSUMER DISCRETIONARY
    'AMZN': 'Consumer Discretionary', 'HD': 'Consumer Discretionary', 'LOW': 'Consumer Discretionary',
    'MCD': 'Consumer Discretionary', 'SBUX': 'Consumer Discretionary', 'NKE': 'Consumer Discretionary',
    'TJX': 'Consumer Discretionary', 'TGT': 'Consumer Discretionary', 'BKNG': 'Consumer Discretionary',
    'YUM': 'Consumer Discretionary', 'LULU': 'Consumer Discretionary', 'ROST': 'Consumer Discretionary',
    'ULTA': 'Consumer Discretionary', 'AZO': 'Consumer Discretionary', 'KMX': 'Consumer Discretionary',
    'ORLY': 'Consumer Discretionary', 'GPC': 'Consumer Discretionary', 'BBY': 'Consumer Discretionary',
    'BWA': 'Consumer Discretionary', 'CCL': 'Consumer Discretionary', 'DECK': 'Consumer Discretionary',
    'DRI': 'Consumer Discretionary', 'DHI': 'Consumer Discretionary', 'DLTR': 'Consumer Discretionary',
    'EXPE': 'Consumer Discretionary', 'HAS': 'Consumer Discretionary', 'LKQ': 'Consumer Discretionary',
    'LEN': 'Consumer Discretionary', 'LVS': 'Consumer Discretionary', 'MGM': 'Consumer Discretionary',
    'MAR': 'Consumer Discretionary', 'MAS': 'Consumer Discretionary', 'MHK': 'Consumer Discretionary',
    'NVR': 'Consumer Discretionary', 'PHM': 'Consumer Discretionary', 'POOL': 'Consumer Discretionary',
    'RL': 'Consumer Discretionary', 'RCL': 'Consumer Discretionary', 'TPR': 'Consumer Discretionary',
    'TSCO': 'Consumer Discretionary', 'WYNN': 'Consumer Discretionary', 'CMG': 'Consumer Discretionary',
    'EBAY': 'Consumer Discretionary', 'F': 'Consumer Discretionary', 'GRMN': 'Consumer Discretionary',
    
    # CONSUMER STAPLES
    'KO': 'Consumer Staples', 'PEP': 'Consumer Staples', 'WMT': 'Consumer Staples',
    'COST': 'Consumer Staples', 'PG': 'Consumer Staples', 'PM': 'Consumer Staples',
    'MO': 'Consumer Staples', 'STZ': 'Consumer Staples', 'CL': 'Consumer Staples',
    'CLX': 'Consumer Staples', 'KMB': 'Consumer Staples', 'CHD': 'Consumer Staples',
    'CPB': 'Consumer Staples', 'CAG': 'Consumer Staples', 'HSY': 'Consumer Staples',
    'HRL': 'Consumer Staples', 'SJM': 'Consumer Staples', 'MKC': 'Consumer Staples',
    'TAP': 'Consumer Staples', 'TSN': 'Consumer Staples', 'K': 'Consumer Staples',
    'KDP': 'Consumer Staples', 'MNST': 'Consumer Staples', 'WBA': 'Consumer Staples',
    'KR': 'Consumer Staples', 'MDLZ': 'Consumer Staples', 'SYY': 'Consumer Staples',
    'ADM': 'Consumer Staples', 'BF-B': 'Consumer Staples', 'EL': 'Consumer Staples',
    'GIS': 'Consumer Staples', 'DG': 'Consumer Staples',
    
    # ENERGY
    'XOM': 'Energy', 'CVX': 'Energy', 'COP': 'Energy', 'SLB': 'Energy',
    'EOG': 'Energy', 'OXY': 'Energy', 'VLO': 'Energy', 'HES': 'Energy',
    'DVN': 'Energy', 'CTRA': 'Energy', 'APA': 'Energy', 'OKE': 'Energy',
    'WMB': 'Energy', 'BKR': 'Energy', 'EQT': 'Energy', 'FCX': 'Energy',
    'TPL': 'Energy', 'MPC': 'Energy',
    
    # FINANCIALS
    'JPM': 'Financials', 'BAC': 'Financials', 'WFC': 'Financials', 'GS': 'Financials',
    'MS': 'Financials', 'C': 'Financials', 'BLK': 'Financials', 'SCHW': 'Financials',
    'USB': 'Financials', 'PNC': 'Financials', 'TFC': 'Financials', 'COF': 'Financials',
    'AIG': 'Financials', 'ALL': 'Financials', 'AFL': 'Financials', 'PRU': 'Financials',
    'MET': 'Financials', 'TRV': 'Financials', 'HIG': 'Financials', 'BRK-B': 'Financials',
    'CB': 'Financials', 'CINF': 'Financials', 'ACGL': 'Financials', 'AON': 'Financials',
    'AJG': 'Financials', 'BRO': 'Financials', 'WRB': 'Financials', 'AIZ': 'Financials',
    'BK': 'Financials', 'BEN': 'Financials', 'IVZ': 'Financials', 'TROW': 'Financials',
    'NDAQ': 'Financials', 'CME': 'Financials', 'ICE': 'Financials', 'NTRS': 'Financials',
    'STT': 'Financials', 'FITB': 'Financials', 'HBAN': 'Financials', 'KEY': 'Financials',
    'MTB': 'Financials', 'RF': 'Financials', 'MCO': 'Financials', 'SPGI': 'Financials',
    'MKTX': 'Financials', 'RJF': 'Financials', 'PFG': 'Financials', 'MSCI': 'Financials',
    'CSGP': 'Financials', 'FICO': 'Financials', 'FI': 'Financials', 'ERIE': 'Financials',
    'GL': 'Financials', 'GPN': 'Financials', 'WTW': 'Financials', 'L': 'Financials',
    'MA': 'Financials', 'V': 'Financials', 'AMP': 'Financials', 'BX': 'Financials',
    'DFS': 'Financials', 'EG': 'Financials', 'EFX': 'Financials', 'FIS': 'Financials',
    'MMC': 'Financials', 'PGR': 'Financials', 'RVTY': 'Financials', 'VRSK': 'Financials',
    
    # HEALTHCARE
    'JNJ': 'Healthcare', 'UNH': 'Healthcare', 'PFE': 'Healthcare', 'ABT': 'Healthcare',
    'TMO': 'Healthcare', 'ABBV': 'Healthcare', 'MRK': 'Healthcare', 'AMGN': 'Healthcare',
    'BMY': 'Healthcare', 'GILD': 'Healthcare', 'CI': 'Healthcare', 'CVS': 'Healthcare',
    'HUM': 'Healthcare', 'ELV': 'Healthcare', 'CNC': 'Healthcare', 'MOH': 'Healthcare',
    'DVA': 'Healthcare', 'UHS': 'Healthcare', 'CAH': 'Healthcare', 'MCK': 'Healthcare',
    'ZBH': 'Healthcare', 'BAX': 'Healthcare', 'BDX': 'Healthcare', 'BSX': 'Healthcare',
    'EW': 'Healthcare', 'HOLX': 'Healthcare', 'ISRG': 'Healthcare', 'IDXX': 'Healthcare',
    'RMD': 'Healthcare', 'SYK': 'Healthcare', 'TECH': 'Healthcare', 'ZTS': 'Healthcare',
    'ALGN': 'Healthcare', 'BIIB': 'Healthcare', 'DXCM': 'Healthcare', 'INCY': 'Healthcare',
    'IFF': 'Healthcare', 'REGN': 'Healthcare', 'VRTX': 'Healthcare', 'A': 'Healthcare',
    'LH': 'Healthcare', 'DGX': 'Healthcare', 'HSIC': 'Healthcare', 'LLY': 'Healthcare',
    'WST': 'Healthcare', 'MDT': 'Healthcare', 'COO': 'Healthcare', 'CRL': 'Healthcare',
    'DHR': 'Healthcare', 'PODD': 'Healthcare', 'VTRS': 'Healthcare', 'HCA': 'Healthcare',
    
    # INDUSTRIALS
    'BA': 'Industrials', 'CAT': 'Industrials', 'DE': 'Industrials', 'GE': 'Industrials',
    'HON': 'Industrials', 'RTX': 'Industrials', 'LMT': 'Industrials', 'NOC': 'Industrials',
    'GD': 'Industrials', 'TXT': 'Industrials', 'TDG': 'Industrials', 'AME': 'Industrials',
    'EMR': 'Industrials', 'ETN': 'Industrials', 'ROK': 'Industrials', 'ITW': 'Industrials',
    'PH': 'Industrials', 'CMI': 'Industrials', 'PCAR': 'Industrials', 'UAL': 'Industrials',
    'DAL': 'Industrials', 'LUV': 'Industrials', 'FDX': 'Industrials', 'UPS': 'Industrials',
    'ODFL': 'Industrials', 'JBHT': 'Industrials', 'CHRW': 'Industrials', 'EXPD': 'Industrials',
    'FAST': 'Industrials', 'GWW': 'Industrials', 'CSX': 'Industrials', 'NSC': 'Industrials',
    'UNP': 'Industrials', 'CPRT': 'Industrials', 'URI': 'Industrials', 'AOS': 'Industrials',
    'DOV': 'Industrials', 'ROL': 'Industrials', 'TT': 'Industrials', 'PWR': 'Industrials',
    'VMC': 'Industrials', 'WAB': 'Industrials', 'WAT': 'Industrials', 'GNRC': 'Industrials',
    'WM': 'Industrials', 'RSG': 'Industrials', 'SNA': 'Industrials', 'SWK': 'Industrials',
    'TFX': 'Industrials', 'BLDR': 'Industrials', 'BR': 'Industrials', 'CTAS': 'Industrials',
    'GEN': 'Industrials', 'HUBB': 'Industrials', 'IEX': 'Industrials', 'JCI': 'Industrials',
    'JBL': 'Industrials', 'LDOS': 'Industrials', 'LHX': 'Industrials', 'MTD': 'Industrials',
    'PNR': 'Industrials', 'ROP': 'Industrials', 'IT': 'Industrials', 'TYC': 'Industrials',
    
    # MATERIALS
    'LIN': 'Materials', 'APD': 'Materials', 'SHW': 'Materials', 'ECL': 'Materials',
    'DD': 'Materials', 'FCX': 'Materials', 'NUE': 'Materials', 'STLD': 'Materials',
    'CF': 'Materials', 'MOS': 'Materials', 'FMC': 'Materials', 'PPG': 'Materials',
    'CE': 'Materials', 'ALB': 'Materials', 'AVY': 'Materials', 'BALL': 'Materials',
    'PKG': 'Materials', 'MLM': 'Materials', 'NEM': 'Materials', 'IP': 'Materials',
    'EMN': 'Materials', 'VMC': 'Materials', 'DOW': 'Materials',
    
    # REAL ESTATE
    'AMT': 'Real Estate', 'PLD': 'Real Estate', 'EQIX': 'Real Estate', 'PSA': 'Real Estate',
    'WELL': 'Real Estate', 'SPG': 'Real Estate', 'EXR': 'Real Estate', 'EQR': 'Real Estate',
    'AVB': 'Real Estate', 'ESS': 'Real Estate', 'UDR': 'Real Estate', 'MAA': 'Real Estate',
    'CPT': 'Real Estate', 'CBRE': 'Real Estate', 'DLR': 'Real Estate', 'REG': 'Real Estate',
    'VTR': 'Real Estate', 'KIM': 'Real Estate', 'IRM': 'Real Estate', 'ARE': 'Real Estate',
    'BXP': 'Real Estate', 'HST': 'Real Estate', 'CCI': 'Real Estate', 'SBAC': 'Real Estate',
    'DOC': 'Real Estate', 'O': 'Real Estate', 'WY': 'Real Estate', 'COR': 'Real Estate',
    'FRT': 'Real Estate',
    
    # UTILITIES
    'NEE': 'Utilities', 'DUK': 'Utilities', 'SO': 'Utilities', 'SRE': 'Utilities',
    'AEP': 'Utilities', 'EXC': 'Utilities', 'XEL': 'Utilities', 'WEC': 'Utilities',
    'ES': 'Utilities', 'ETR': 'Utilities', 'EVRG': 'Utilities', 'FE': 'Utilities',
    'NI': 'Utilities', 'PNW': 'Utilities', 'PPL': 'Utilities', 'AEE': 'Utilities',
    'ATO': 'Utilities', 'CMS': 'Utilities', 'CNP': 'Utilities', 'ED': 'Utilities',
    'EIX': 'Utilities', 'LNT': 'Utilities', 'NRG': 'Utilities', 'PCG': 'Utilities',
    'PEG': 'Utilities', 'D': 'Utilities', 'DTE': 'Utilities', 'AWK': 'Utilities',
    'AES': 'Utilities',
}

# Create DataFrame with all tickers and their sectors
sector_df = pd.DataFrame({
    'Symbol': all_tickers,
    'Sector': [SECTOR_MAPPING.get(ticker, 'Unknown') for ticker in all_tickers]
})

# Check how many we mapped
mapped_count = (sector_df['Sector'] != 'Unknown').sum()
print(f"✅ Mapped {mapped_count}/{len(all_tickers)} tickers to sectors")
print(f"⚠️  {len(all_tickers) - mapped_count} tickers need manual mapping")

# Show unmapped tickers
unmapped = sector_df[sector_df['Sector'] == 'Unknown']['Symbol'].tolist()
if unmapped:
    print(f"\nUnmapped tickers: {unmapped}")

# Show sector distribution
print(f"\n📊 Sector Distribution:")
print(sector_df['Sector'].value_counts().sort_values(ascending=False))

# Save to CSV
output_path = "./dataset/stock_sectors.csv"
sector_df.to_csv(output_path, index=False)
print(f"\n💾 Saved sector mapping to: {output_path}")

# Create a lookup dictionary for easy access
stock_sectors = dict(zip(sector_df['Symbol'], sector_df['Sector']))

# Display first few rows
print("\n📋 First 10 rows:")
display(sector_df.head(10))


✅ Mapped 415/432 tickers to sectors
⚠️  17 tickers need manual mapping

Unmapped tickers: ['WDC', 'MMM', 'CTSH', 'ADP', 'ACN', 'BG', 'AXP', 'AXON', 'HAL', 'GLW', 'JNPR', 'J', 'IPG', 'DPZ', 'FSLR', 'FFIV', 'FDS']

📊 Sector Distribution:
Sector
Financials                68
Industrials               62
Healthcare                49
Information Technology    47
Consumer Discretionary    45
Consumer Staples          32
Real Estate               29
Utilities                 29
Materials                 22
Unknown                   17
Communication Services    16
Energy                    16
Name: count, dtype: int64

💾 Saved sector mapping to: ./dataset/stock_sectors.csv

📋 First 10 rows:


,Symbol,Sector
0,GNRC,Industrials
1,CHTR,Communication Services
2,MTCH,Communication Services
3,NDAQ,Financials
4,WFC,Financials
5,WM,Industrials
6,WELL,Real Estate
7,WMB,Energy
8,MU,Information Technology
9,NDSN,Information Technology


In [17]:
# ============================================================================
# STRATEGIC SUBSET SELECTION: 40 stocks for pairs trading
# Now using actual sector mapping from Cell 2
# ============================================================================

# Load sector mapping (created in previous cell)
# If CSV doesn't exist, use the dictionary from Cell 2, otherwise load from CSV
import os
if os.path.exists("./dataset/stock_sectors.csv"):
    sector_df = pd.read_csv("./dataset/stock_sectors.csv")
    stock_sectors = dict(zip(sector_df['Symbol'], sector_df['Sector']))
else:
    # Fallback: use the stock_sectors dictionary from Cell 2 if it exists
    # (this requires Cell 2 to be run first)
    if 'stock_sectors' in globals():
        stock_sectors = globals()['stock_sectors']
    else:
        print("⚠️  Warning: stock_sectors.csv not found. Please run Cell 2 first to create sector mapping.")
        stock_sectors = {}

SELECTION_METHOD = "sector_based"  # Options: "sector_based", "manual_known_pairs", "data_quality", "correlation"

if SELECTION_METHOD == "sector_based":
    # Option 1: Select stocks from specific sectors that work well for pairs trading
    # Focus on sectors with many similar companies (better cointegration chances)
    
    # Priority sectors for pairs trading (well-known for cointegration)
    target_sectors = [
        'Financials',           # Banks, insurance (JPM/BAC, etc.)
        'Consumer Staples',     # Classic pairs (KO/PEP)
        'Healthcare',          # Stable, often cointegrated
        'Industrials',         # Similar business models
        'Information Technology', # Tech giants
    ]
    
    # Get stocks from target sectors
    sector_stocks = []
    for sector in target_sectors:
        stocks_in_sector = [ticker for ticker, sec in stock_sectors.items() if sec == sector and ticker in prices.columns]
        sector_stocks.extend(stocks_in_sector)
    
    # Select ~8 stocks from each sector (to get ~40 total)
    selected_tickers = []
    stocks_per_sector = 8
    
    for sector in target_sectors:
        stocks_in_sector = [ticker for ticker, sec in stock_sectors.items() if sec == sector and ticker in prices.columns]
        # Prioritize by data quality (less missing data)
        missing_counts = prices[stocks_in_sector].isna().sum().sort_values()
        selected_tickers.extend(missing_counts.head(stocks_per_sector).index.tolist())
        if len(selected_tickers) >= 40:
            break
    
    # Trim to exactly 40
    selected_tickers = selected_tickers[:40]
    
elif SELECTION_METHOD == "manual_known_pairs":
    # Option 2: Manual selection of well-known pairs trading candidates
    strategic_tickers = [
        'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'ORCL', 'CSCO',
        'JPM', 'BAC', 'WFC', 'C', 'GS', 'MS',
        'KO', 'PEP', 'WMT', 'TGT', 'HD', 'LOW', 'MCD', 'SBUX',
        'JNJ', 'PFE', 'UNH', 'ABT', 'MRK', 'TMO',
        'XOM', 'CVX', 'SLB',
        'CAT', 'DE', 'GE', 'HON',
        'AIG', 'BRK-B', 'BLK',
        'VZ', 'T'
    ]
    
    available_tickers = prices.columns.tolist()
    selected_tickers = [t for t in strategic_tickers if t in available_tickers]
    
    if len(selected_tickers) < 40:
        additional_candidates = [t for t in available_tickers if t not in selected_tickers]
        missing_counts = prices[additional_candidates].isna().sum().sort_values()
        selected_tickers.extend(missing_counts.head(40 - len(selected_tickers)).index.tolist())
        selected_tickers = selected_tickers[:40]

elif SELECTION_METHOD == "data_quality":
    # Option 3: Select stocks based on data quality
    missing_counts = prices.isna().sum().sort_values()
    selected_tickers = missing_counts.head(40).index.tolist()
    
elif SELECTION_METHOD == "correlation":
    # Option 4: Select stocks that are highly correlated
    returns = prices.pct_change().dropna()
    corr_matrix = returns.corr().abs()
    avg_corr = corr_matrix.mean().sort_values(ascending=False)
    selected_tickers = avg_corr.head(40).index.tolist()

# Subset the prices dataframe
prices_subset = prices[selected_tickers].copy()

# Show sector breakdown of selected stocks
selected_sectors = [stock_sectors.get(t, 'Unknown') for t in selected_tickers]
sector_counts = pd.Series(selected_sectors).value_counts()

print(f"Selection method: {SELECTION_METHOD}")
print(f"Selected {len(selected_tickers)} stocks for pairs trading")
print(f"This will test {len(list(itertools.combinations(selected_tickers, 2)))} pairs")
print(f"\n📊 Sector breakdown of selected stocks:")
for sector, count in sector_counts.items():
    print(f"  {sector}: {count} stocks")

print(f"\nSelected tickers: {sorted(selected_tickers)}")


Selection method: sector_based
Selected 40 stocks for pairs trading
This will test 780 pairs

📊 Sector breakdown of selected stocks:
  Financials: 8 stocks
  Consumer Staples: 8 stocks
  Healthcare: 8 stocks
  Industrials: 8 stocks
  Information Technology: 8 stocks

Selected tickers: ['ABT', 'ADM', 'AJG', 'ALGN', 'ALL', 'AMAT', 'AMD', 'AMGN', 'AMP', 'BDX', 'BF-B', 'BIIB', 'BLK', 'BMY', 'BR', 'CDNS', 'CME', 'CMI', 'CNC', 'COF', 'CPRT', 'CRM', 'CSCO', 'CSGP', 'CSX', 'CTAS', 'DAL', 'DE', 'GIS', 'GNRC', 'HRL', 'HSY', 'KDP', 'KMB', 'MRK', 'MU', 'NDAQ', 'PTC', 'QCOM', 'WMT']


In [18]:
# OPTIONAL: Quick test with just 10 stocks first (45 pairs) - runs in seconds
# Uncomment the lines below to run a quick test before processing all 40 stocks

# TEST_MODE = True  # Set to True for quick test
# if TEST_MODE:
#     test_tickers = selected_tickers[:10]
#     print(f"TEST MODE: Using {len(test_tickers)} stocks = {len(list(itertools.combinations(test_tickers, 2)))} pairs")
#     print(f"Test tickers: {test_tickers}")
#     selected_tickers = test_tickers
#     prices_subset = prices[selected_tickers].copy()


In [19]:
def run_coint(pair):
    a, b = pair
    s1 = prices[a].dropna()
    s2 = prices[b].dropna()
    _, pvalue, _ = coint(s1, s2)
    return (pair, pvalue)

In [20]:
from pathos.multiprocessing import ProcessingPool as Pool
from statsmodels.tsa.stattools import coint
import itertools
from tqdm import tqdm
import pandas as pd
import numpy as np

# Use the subset of tickers we selected
tickers = selected_tickers
pairs = list(itertools.combinations(tickers, 2))

print(f"Testing {len(pairs)} pairs from {len(tickers)} stocks...")

def run_coint(pair):
    """Run cointegration test with error handling"""
    try:
        a, b = pair
        s1 = prices_subset[a].dropna()
        s2 = prices_subset[b].dropna()
        
        # Skip if either series is too short (minimum 100 observations for reliable test)
        if len(s1) < 100 or len(s2) < 100:
            return (pair, np.nan)
        
        # Align series by date to ensure same length
        common_dates = s1.index.intersection(s2.index)
        if len(common_dates) < 100:
            return (pair, np.nan)
        
        s1_aligned = s1.loc[common_dates]
        s2_aligned = s2.loc[common_dates]
        
        # Run cointegration test
        _, pvalue, _ = coint(s1_aligned, s2_aligned)
        return (pair, pvalue)
    except Exception as e:
        # Return NaN if test fails (e.g., insufficient data, numerical issues)
        return (pair, np.nan)

# Use multiprocessing with specified number of processes
# Adjust based on your CPU cores (4-8 is usually good, but use fewer if you have limited RAM)
n_processes = 4  # You can increase this if you have more CPU cores
pool = Pool(n_processes)

results = []
# Use tqdm.imap with error handling
try:
    for res in tqdm(
        pool.imap(run_coint, pairs), 
        total=len(pairs), 
        desc="Cointegration tests",
        mininterval=1.0  # Update progress bar at least every second
    ):
        results.append(res)
except KeyboardInterrupt:
    print("\nInterrupted by user. Saving partial results...")
finally:
    pool.close()
    pool.join()

# Convert to DataFrame and filter out NaN results
coint_results = pd.DataFrame(results, columns=["pair", "pvalue"])
coint_results = coint_results.dropna(subset=["pvalue"]).sort_values("pvalue")

print(f"\n✓ Completed {len(coint_results)}/{len(pairs)} tests successfully")
print(f"\nTop 10 most cointegrated pairs (lowest p-values):")
display(coint_results.head(10))

# Summary statistics
print(f"\nSummary:")
print(f"  - Total pairs tested: {len(pairs)}")
print(f"  - Successful tests: {len(coint_results)}")
print(f"  - Failed tests: {len(pairs) - len(coint_results)}")
print(f"  - Pairs with p-value < 0.05: {len(coint_results[coint_results['pvalue'] < 0.05])}")
print(f"  - Pairs with p-value < 0.01: {len(coint_results[coint_results['pvalue'] < 0.01])}")

Testing 780 pairs from 40 stocks...


Cointegration tests: 100%|██████████| 780/780 [2:08:29<00:00,  9.88s/it]  


✓ Completed 780/780 tests successfully

Top 10 most cointegrated pairs (lowest p-values):


,pair,pvalue
279,"(AJG, CDNS)",0.000035
73,"(CME, CSCO)",0.000055
756,"(MU, CRM)",0.000156
694,"(CMI, MU)",0.001473
681,"(BR, MU)",0.002029
301,"(WMT, CMI)",0.003729
779,"(AMD, AMAT)",0.003966
177,"(BLK, MU)",0.005147
753,"(MU, PTC)",0.005709
95,"(CSGP, ABT)",0.006102



Summary:
  - Total pairs tested: 780
  - Successful tests: 780
  - Failed tests: 0
  - Pairs with p-value < 0.05: 51
  - Pairs with p-value < 0.01: 18


In [23]:
# ============================================================================
# ANALYZE SECTORS FOR HIGHLY COINTEGRATED PAIRS (p-value < 0.01)
# ============================================================================

import os

# Load sector mapping
if os.path.exists("./dataset/stock_sectors.csv"):
    sector_df = pd.read_csv("./dataset/stock_sectors.csv")
    stock_sectors = dict(zip(sector_df['Symbol'], sector_df['Sector']))
else:
    # Use the dictionary from Cell 2 if available
    stock_sectors = globals().get('stock_sectors', {})

# Filter highly cointegrated pairs (p-value < 0.01)
highly_cointegrated = coint_results[coint_results['pvalue'] < 0.01].copy()

print(f"📊 Analyzing {len(highly_cointegrated)} highly cointegrated pairs (p-value < 0.01)\n")
print("=" * 80)

# Extract sectors for each pair
pair_analysis = []
for idx, row in highly_cointegrated.iterrows():
    pair = row['pair']
    ticker1, ticker2 = pair
    sector1 = stock_sectors.get(ticker1, 'Unknown')
    sector2 = stock_sectors.get(ticker2, 'Unknown')
    
    # Determine if same sector or cross-sector
    same_sector = sector1 == sector2
    
    pair_analysis.append({
        'Ticker1': ticker1,
        'Sector1': sector1,
        'Ticker2': ticker2,
        'Sector2': sector2,
        'Same_Sector': same_sector,
        'pvalue': row['pvalue']
    })

# Create DataFrame for better display
pair_df = pd.DataFrame(pair_analysis).sort_values('pvalue')

print("\n🎯 Highly Cointegrated Pairs (p-value < 0.01):")
print("=" * 80)

for idx, row in pair_df.iterrows():
    same_sector_marker = "✓" if row['Same_Sector'] else "✗"
    print(f"\n{same_sector_marker} {row['Ticker1']:6s} ({row['Sector1']:25s})  ↔  {row['Ticker2']:6s} ({row['Sector2']:25s})")
    print(f"   p-value: {row['pvalue']:.6f}")

# Statistics
same_sector_count = pair_df['Same_Sector'].sum()
cross_sector_count = len(pair_df) - same_sector_count
same_sector_pct = (same_sector_count / len(pair_df)) * 100

print("\n" + "=" * 80)
print("\n📈 SECTOR ANALYSIS STATISTICS:")
print("-" * 80)
print(f"Total highly cointegrated pairs: {len(pair_df)}")
print(f"Same-sector pairs: {same_sector_count} ({same_sector_pct:.1f}%)")
print(f"Cross-sector pairs: {cross_sector_count} ({100 - same_sector_pct:.1f}%)")

# Sector combinations
print("\n📋 Sector Combinations:")
print("-" * 80)

# Same sector pairs
same_sector_pairs = pair_df[pair_df['Same_Sector'] == True]
if len(same_sector_pairs) > 0:
    print("\n✅ Same-Sector Pairs:")
    sector_counts = same_sector_pairs['Sector1'].value_counts()
    for sector, count in sector_counts.items():
        print(f"  {sector}: {count} pair(s)")
        # Show specific pairs in this sector
        sector_pairs = same_sector_pairs[same_sector_pairs['Sector1'] == sector]
        for _, pair_row in sector_pairs.iterrows():
            print(f"    • {pair_row['Ticker1']} ↔ {pair_row['Ticker2']} (p={pair_row['pvalue']:.6f})")

# Cross sector pairs
cross_sector_pairs = pair_df[pair_df['Same_Sector'] == False]
if len(cross_sector_pairs) > 0:
    print("\n❌ Cross-Sector Pairs:")
    for _, pair_row in cross_sector_pairs.iterrows():
        print(f"  • {pair_row['Ticker1']} ({pair_row['Sector1']}) ↔ {pair_row['Ticker2']} ({pair_row['Sector2']})")
        print(f"    p-value: {pair_row['pvalue']:.6f}")

print("\n" + "=" * 80)

# Display as DataFrame for easy export
print("\n📋 Complete DataFrame:")
display(pair_df)


📊 Analyzing 18 highly cointegrated pairs (p-value < 0.01)


🎯 Highly Cointegrated Pairs (p-value < 0.01):

✗ AJG    (Financials               )  ↔  CDNS   (Information Technology   )
   p-value: 0.000035

✗ CME    (Financials               )  ↔  CSCO   (Information Technology   )
   p-value: 0.000055

✓ MU     (Information Technology   )  ↔  CRM    (Information Technology   )
   p-value: 0.000156

✗ CMI    (Industrials              )  ↔  MU     (Information Technology   )
   p-value: 0.001473

✗ BR     (Industrials              )  ↔  MU     (Information Technology   )
   p-value: 0.002029

✗ WMT    (Consumer Staples         )  ↔  CMI    (Industrials              )
   p-value: 0.003729

✓ AMD    (Information Technology   )  ↔  AMAT   (Information Technology   )
   p-value: 0.003966

✗ BLK    (Financials               )  ↔  MU     (Information Technology   )
   p-value: 0.005147

✓ MU     (Information Technology   )  ↔  PTC    (Information Technology   )
   p-value: 0.005709

✗ CSGP   (F

,Ticker1,Sector1,Ticker2,Sector2,Same_Sector,pvalue
0,AJG,Financials,CDNS,Information Technology,False,0.000035
1,CME,Financials,CSCO,Information Technology,False,0.000055
2,MU,Information Technology,CRM,Information Technology,True,0.000156
3,CMI,Industrials,MU,Information Technology,False,0.001473
4,BR,Industrials,MU,Information Technology,False,0.002029
5,WMT,Consumer Staples,CMI,Industrials,False,0.003729
6,AMD,Information Technology,AMAT,Information Technology,True,0.003966
7,BLK,Financials,MU,Information Technology,False,0.005147
8,MU,Information Technology,PTC,Information Technology,True,0.005709
9,CSGP,Financials,ABT,Healthcare,False,0.006102


In [1]:
pair_df

NameError: name 'pair_df' is not defined